### *WEB SCRAPING*


#### **GOAL**:
1. Scraping job listings from a job board remoteok.com for remote jobs using the API and JSON
2. Extracting the Details:
   * Job Title
   * Company Name
   * Location
   * Minimum Salary
   * Maximum Salary
   * Date Posted
3. Saving the data in a CSV file:

**Important Points:**
 * The URL (https://remoteok.com/api) is the API endpoint that provides job listings in JSON format.
 * The headers dictionary includes a User-Agent string to mimic a request from a web browser (useful to avoid being blocked by the server).
 * response stores the HTTP response, which is in JSON (JavaScript Object Notation)format containing job listings.
 * The response.json() method converts the JSON response into a Python dictionary/list.
 * The response contains some metadata as the first element, so [1:] is used to skip this and extract the actual job listings.
   
Using, all the above we are Extracting all the Details we need and then saving the details into .csv file.

In [9]:
import requests
import pandas as pd
import re

# API endpoint
url = "https://remoteok.com/api"

# Send request
headers = {
    "User-Agent": "Mozilla/5.0"
}
response = requests.get(url, headers=headers)

# Skip the metadata at index 0
jobs = response.json()[1:]

# Extract relevant data with handling for missing fields
data = []
for job in jobs:
    title = job.get('position', '')
    company = job.get('company', '')
    
    # Handle empty or missing location
    location = job.get('location')
    if not location:
        location = 'NA'
    
    # Handle salary ranges
    salary_min = job.get('salary_min')
    salary_max = job.get('salary_max')

    salary_min = salary_min if salary_min else 'NA'
    salary_max = salary_max if salary_max else 'NA'
    
    # Get the date and apply regex to format it
    date_posted = job.get('date', '')
    # Use regex to replace 'T' with a space and remove the timezone part
    formatted_date = re.sub(r'T', ' ', date_posted)  # Replace T with space
    formatted_date = re.sub(r'\+.*', '', formatted_date)  # Remove timezone part (e.g., +00:00)
    
    # Set up the job data dictionary
    data.append({
        'Job Title': title,
        'Company': company,
        'Location': location,
        'Minimum Salary (Rupees)': salary_min,
        'Maximum Salary (Rupees)': salary_max,
        'Date Posted': formatted_date
    })

# Create DataFrame
df = pd.DataFrame(data)

# Limit to first 100 jobs
df = df.head(100)

# Save to CSV
df.to_csv("Remoteok_Jobs.csv", index=False)

